In [4]:
# 7_add_sipher_pop_row_counts.ipynb
#
# Adds SIPHER population row counts to the UKHLS respondent table from step 5.
#
# Each UKHLS respondent (pidp) appears multiple times in the SIPHER synthetic
# microdata. The count (n_sipher_rows) is their effective population weight.
#
# Inputs:
#   data/1_pickle_sipher/sipher_optimized.pkl        — pidp × synthetic_zone (52M rows)
#   data/5_add_nl_strings/k_with_nl_profile.pkl      — UKHLS respondents + NL profile
#
# Outputs (data/7_add_sipher_pop_row_counts/):
#   k_with_sipher_row_counts.pkl
#   k_with_sipher_row_counts.csv

import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import data_pipeline.config_variables as _cv
_cv.reload_config_variables()
from data_pipeline.config_variables import DATA_FOLDER

import pandas as pd

SIPHER_PKL = Path(f"../{DATA_FOLDER}/1_pickle_sipher/sipher_optimized.pkl")
NL_PKL     = Path(f"../{DATA_FOLDER}/5_add_nl_strings/k_with_nl_profile.pkl")
OUT_DIR    = Path(f"../{DATA_FOLDER}/7_add_sipher_pop_row_counts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PKL = OUT_DIR / "k_with_sipher_row_counts.pkl"
OUT_CSV = OUT_DIR / "k_with_sipher_row_counts.csv"

for p in (SIPHER_PKL, NL_PKL):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found")

# ── 1. Count pidp occurrences in SIPHER ───────────────────────────────────────
print(f"Reading {SIPHER_PKL.name} ...")
df_sipher = pd.read_pickle(SIPHER_PKL)
df_sipher["pidp"] = pd.to_numeric(df_sipher["pidp"], errors="coerce").astype("int64")
print(f"  {len(df_sipher):,} rows, {df_sipher['pidp'].nunique():,} unique pidps")

pidp_counts = (
    df_sipher.groupby("pidp", sort=False)
    .size()
    .rename("n_sipher_rows")
    .reset_index()
)
del df_sipher

# ── 2. Load step-5 table ──────────────────────────────────────────────────────
print(f"\nReading {NL_PKL.name} ...")
df = pd.read_pickle(NL_PKL)
df["pidp"] = pd.to_numeric(df["pidp"], errors="coerce").astype("int64")
print(f"  {len(df):,} respondents × {len(df.columns)} cols")

# ── 3. Append count column ────────────────────────────────────────────────────
df = df.merge(pidp_counts, on="pidp", how="left")
df["n_sipher_rows"] = df["n_sipher_rows"].fillna(0).astype("int64")

n_matched = (df["n_sipher_rows"] > 0).sum()
print(f"\n  {n_matched:,} / {len(df):,} respondents matched in SIPHER")
s = df["n_sipher_rows"]
print(f"  n_sipher_rows — min={s.min()}  max={s.max()}  mean={s.mean():.1f}")

# ── 4. Save ───────────────────────────────────────────────────────────────────
df.to_pickle(OUT_PKL)
df.to_csv(OUT_CSV, index=False)
print(f"\nSaved {len(df):,} rows × {len(df.columns)} cols")
print(f"  → {OUT_PKL}")
print(f"  → {OUT_CSV}")


Reading sipher_optimized.pkl ...
  52,853,971 rows, 27,330 unique pidps

Reading k_with_nl_profile.pkl ...
  27,330 respondents × 42 cols

  27,330 / 27,330 respondents matched in SIPHER
  n_sipher_rows — min=255  max=43858  mean=1933.9

Saved 27,330 rows × 43 cols
  → ../data/7_add_sipher_pop_row_counts/k_with_sipher_row_counts.pkl
  → ../data/7_add_sipher_pop_row_counts/k_with_sipher_row_counts.csv
